<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
# ============================================================
# ML-07 / W04 — BASELINE ACTION SCORE
# STEP 0: LOAD FINAL FEATURE DATASET
# ============================================================

import pandas as pd
import numpy as np

print("=" * 70)
print("W04 — BASELINE ACTION SCORE")
print("STEP 0: LOAD FINAL FEATURE DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Load the 20-column processed Parquet dataset
# ------------------------------------------------------------

df_features = pd.read_parquet(
    "/final_features_clean.parquet"
)

# Work on a copy
df_baseline = df_features.copy()

# ------------------------------------------------------------
# Basic inspection
# ------------------------------------------------------------

print("\nDataset loaded successfully.")

print(f"Rows    : {len(df_baseline):,}")
print(f"Columns : {len(df_baseline.columns)}")

print("\nColumns:")
for i, col in enumerate(df_baseline.columns, start=1):
    print(f"{i:2}. {col}")

print("\nShape:")
print(df_baseline.shape)

print("\nFirst 5 rows:")
display(df_baseline.head())

W04 — BASELINE ACTION SCORE
STEP 0: LOAD FINAL FEATURE DATASET

Dataset loaded successfully.
Rows    : 2,871,202
Columns : 19

Columns:
 1. client_hash_id
 2. content_hash_id
 3. month
 4. gsc_clicks
 5. gsc_impressions
 6. gsc_avg_position
 7. ga4_total_engagement_sec
 8. sessions_organic
 9. sessions_ai
10. missing_count
11. gsc_avg_position_missing
12. ga4_total_engagement_sec_missing
13. sessions_organic_missing
14. sessions_ai_missing
15. ai_other_missing
16. ctr
17. sec_per_click
18. ai_share
19. engagement_per_organic_session

Shape:
(2871202, 19)

First 5 rows:


,client_hash_id,content_hash_id,month,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_total_engagement_sec,sessions_organic,sessions_ai,missing_count,gsc_avg_position_missing,ga4_total_engagement_sec_missing,sessions_organic_missing,sessions_ai_missing,ai_other_missing,ctr,sec_per_click,ai_share,engagement_per_organic_session
0,client_9958f0a7ae1df715,content_000005d4ced12088,2025-03-01,0.0,7.0,9.333333,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
1,client_9958f0a7ae1df715,content_000005d4ced12088,2025-04-01,1.0,146.0,35.762918,0.0,0.0,0.0,0,0,0,0,0,0,0.006849,0.0,0.0,0.0
2,client_9958f0a7ae1df715,content_000005d4ced12088,2025-05-01,0.0,257.0,38.982641,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
3,client_9958f0a7ae1df715,content_000005d4ced12088,2025-06-01,0.0,139.0,37.522978,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0
4,client_9958f0a7ae1df715,content_000005d4ced12088,2025-07-01,0.0,254.0,35.843110,0.0,0.0,0.0,0,0,0,0,0,0,0.000000,0.0,0.0,0.0


**SIGNAL TEST #1** : Flyrank Staleness Signal

In [20]:
# ============================================================
# STALENESS SIGNAL — 90 DAY ELIGIBILITY + EXACT BUCKETS
# ============================================================

df_staleness = df_baseline.copy()

df_staleness["month"] = pd.to_datetime(
    df_staleness["month"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. First and last observed month for each page
# ------------------------------------------------------------

first_month = (
    df_staleness
    .groupby("content_hash_id")["month"]
    .transform("min")
)

last_month = (
    df_staleness
    .groupby("content_hash_id")["month"]
    .transform("max")
)

# Observed history
df_staleness["content_age_days"] = (
    last_month - first_month
).dt.days

# ------------------------------------------------------------
# 2. Keep EXACTLY 90 days and above
# ------------------------------------------------------------

df_staleness_90 = df_staleness[
    df_staleness["content_age_days"] >= 90
].copy()

print("=" * 70)
print("90-DAY ELIGIBILITY CHECK")
print("=" * 70)

print(
    f"Rows before filter : {len(df_staleness):,}"
)

print(
    f"Rows after filter  : {len(df_staleness_90):,}"
)

print(
    f"Rows removed       : "
    f"{len(df_staleness) - len(df_staleness_90):,}"
)

# Explicitly verify exact 90-day observations
exact_90 = (
    df_staleness_90["content_age_days"] == 90
).sum()

print(
    f"\nExact 90-day observations included: "
    f"{exact_90:,}"
)

# ------------------------------------------------------------
# 3. Create buckets WITHOUT pd.cut boundary ambiguity
# ------------------------------------------------------------

df_staleness_90["age_bucket"] = np.select(
    [
        df_staleness_90["content_age_days"] <= 180,
        df_staleness_90["content_age_days"] <= 365,
        df_staleness_90["content_age_days"] > 365
    ],
    [
        "90-180 days",
        "181-365 days",
        "365+ days"
    ],
    default="unknown"
)

# ------------------------------------------------------------
# 4. Check bucket distribution
# ------------------------------------------------------------

bucket_summary = (
    df_staleness_90
    .groupby("age_bucket", sort=False)
    .size()
    .reset_index(name="observations")
)

bucket_summary["percentage"] = (
    bucket_summary["observations"]
    / len(df_staleness_90)
    * 100
).round(2)

print("\n" + "=" * 70)
print("AGE BUCKET DISTRIBUTION")
print("=" * 70)

display(bucket_summary)

# ------------------------------------------------------------
# 5. Verify no eligible observation is lost
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BOUNDARY CHECK")
print("=" * 70)

print(
    "Minimum included age:",
    df_staleness_90["content_age_days"].min()
)

print(
    "Exact 90-day included:",
    exact_90 > 0
)

print(
    "Unknown bucket rows:",
    (df_staleness_90["age_bucket"] == "unknown").sum()
)

90-DAY ELIGIBILITY CHECK
Rows before filter : 2,871,202
Rows after filter  : 2,705,303
Rows removed       : 165,899

Exact 90-day observations included: 0

AGE BUCKET DISTRIBUTION


,age_bucket,observations,percentage
0,365+ days,244525,9.04
1,181-365 days,1939160,71.68
2,90-180 days,521618,19.28



BOUNDARY CHECK
Minimum included age: 92
Exact 90-day included: False
Unknown bucket rows: 0


In [21]:
# ============================================================
# FINAL STALENESS DECAY RATE TEST
# Uses df_staleness_90 created in previous block
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------

df_decay_test = df_staleness_90.copy()

df_decay_test["month"] = pd.to_datetime(
    df_decay_test["month"],
    errors="coerce"
)

df_decay_test = df_decay_test.sort_values(
    ["content_hash_id", "month"]
).reset_index(drop=True)


# ------------------------------------------------------------
# 2. Future 3-month impression value
# ------------------------------------------------------------

df_decay_test["future_impressions"] = (
    df_decay_test
    .groupby("content_hash_id")["gsc_impressions"]
    .shift(-3)
)


# ------------------------------------------------------------
# 3. Future impression change %
# ------------------------------------------------------------

df_decay_test["future_impression_change_pct"] = np.where(
    (
        df_decay_test["gsc_impressions"] > 0
        & df_decay_test["future_impressions"].notna()
    ),
    (
        (
            df_decay_test["future_impressions"]
            - df_decay_test["gsc_impressions"]
        )
        / df_decay_test["gsc_impressions"]
    ) * 100,
    np.nan
)


# ------------------------------------------------------------
# 4. Define decay
# ------------------------------------------------------------

df_decay_test["decay"] = (
    df_decay_test["future_impression_change_pct"] <= -20
)


# ------------------------------------------------------------
# 5. Remove rows without future 3-month value
# ------------------------------------------------------------

df_decay_test = df_decay_test[
    df_decay_test["future_impression_change_pct"].notna()
].copy()


# ============================================================
# 6. DECAY RATE BY AGE BUCKET
# ============================================================

age_order = [
    "90-180 days",
    "181-365 days",
    "365+ days"
]

final_check = (
    df_decay_test
    .groupby("age_bucket", observed=True)
    .agg(
        observations=("decay", "size"),
        decay_cases=("decay", "sum"),
        decay_rate=("decay", "mean"),
        avg_future_impression_change_pct=(
            "future_impression_change_pct",
            "mean"
        )
    )
    .reindex(age_order)
    .reset_index()
)


# ------------------------------------------------------------
# 7. Convert to percentages
# ------------------------------------------------------------

final_check["decay_rate"] = (
    final_check["decay_rate"] * 100
).round(2)

final_check["avg_future_impression_change_pct"] = (
    final_check["avg_future_impression_change_pct"]
    .round(2)
)


# ============================================================
# 8. DISPLAY
# ============================================================

print("=" * 70)
print("FINAL STALENESS SIGNAL VERIFICATION")
print("=" * 70)

display(final_check)


# ============================================================
# 9. VERDICT
# ============================================================

rates = (
    final_check["decay_rate"]
    .dropna()
    .tolist()
)

print("\nDecay rates in age order:")
print(rates)


if len(rates) < 2:

    verdict = "INSUFFICIENT DATA"

elif all(
    rates[i] < rates[i + 1]
    for i in range(len(rates) - 1)
):

    verdict = "CONFIRMED"

elif all(
    rates[i] > rates[i + 1]
    for i in range(len(rates) - 1)
):

    verdict = "OPPOSITE"

else:

    verdict = "MIXED"


print("\n" + "=" * 70)
print("FINAL VERDICT:", verdict)
print("=" * 70)

FINAL STALENESS SIGNAL VERIFICATION


,age_bucket,observations,decay_cases,decay_rate,avg_future_impression_change_pct
0,90-180 days,103259,64667,62.63,727.89
1,181-365 days,588405,274642,46.68,780.78
2,365+ days,188441,73565,39.04,391.70



Decay rates in age order:
[62.63, 46.68, 39.04]

FINAL VERDICT: OPPOSITE


**Signal Test # 2**: CTR VS decay

In [24]:
import pandas as pd
import numpy as np

# ================================================================
# SIGNAL TEST #2 — CTR vs FUTURE DECAY
# ================================================================

# Use your current final dataframe
df_test = df_features.copy()

# Make sure month is datetime
df_test["month"] = pd.to_datetime(df_test["month"])

# Sort page history
df_test = df_test.sort_values(
    ["client_hash_id", "content_hash_id", "month"]
).reset_index(drop=True)


# ================================================================
# 1. CREATE FUTURE 3-MONTH IMPRESSION CHANGE
# ================================================================

# Future impression = impression after 3 months
df_test["future_impressions_3m"] = (
    df_test.groupby(["client_hash_id", "content_hash_id"])["gsc_impressions"]
    .shift(-3)
)

# Current impressions
df_test["current_impressions"] = df_test["gsc_impressions"]


# Future percentage change
# Only calculate where current impressions > 0
df_test["future_impression_change_pct"] = np.where(
    df_test["current_impressions"] > 0,
    (
        (
            df_test["future_impressions_3m"]
            - df_test["current_impressions"]
        )
        / df_test["current_impressions"]
    ) * 100,
    np.nan
)


# ================================================================
# 2. KEEP ONLY OBSERVATIONS WITH A REAL 3-MONTH FUTURE
# ================================================================

df_ctr_test = df_test[
    df_test["future_impression_change_pct"].notna()
].copy()


# ================================================================
# 3. CREATE CTR BUCKETS
# ================================================================

# CTR is already available as a derived feature.
# Convert to percentage for easier interpretation.

df_ctr_test["ctr_percent"] = df_ctr_test["ctr"] * 100

def ctr_bucket(ctr):
    if pd.isna(ctr):
        return np.nan
    elif ctr < 0.5:
        return "Very Low (<0.5%)"
    elif ctr < 1.0:
        return "Low (0.5–1%)"
    elif ctr < 2.0:
        return "Medium (1–2%)"
    elif ctr < 5.0:
        return "High (2–5%)"
    else:
        return "Very High (5%+)"

df_ctr_test["ctr_bucket"] = df_ctr_test["ctr_percent"].apply(ctr_bucket)


# ================================================================
# 4. DEFINE FUTURE DECAY
# ================================================================

# Future decay = impressions decrease by 20% or more
df_ctr_test["future_decay"] = (
    df_ctr_test["future_impression_change_pct"] <= -20
)


# ================================================================
# 5. BUCKET SUMMARY
# ================================================================

bucket_order = [
    "Very Low (<0.5%)",
    "Low (0.5–1%)",
    "Medium (1–2%)",
    "High (2–5%)",
    "Very High (5%+)"
]

ctr_results = (
    df_ctr_test
    .groupby("ctr_bucket", observed=True)
    .agg(
        observations=("future_decay", "size"),
        decay_cases=("future_decay", "sum"),
        avg_future_impression_change_pct=(
            "future_impression_change_pct",
            "mean"
        ),
        median_future_impression_change_pct=(
            "future_impression_change_pct",
            "median"
        )
    )
    .reindex(bucket_order)
    .reset_index()
)

ctr_results["decay_rate"] = (
    ctr_results["decay_cases"]
    / ctr_results["observations"]
    * 100
)


# ================================================================
# 6. DISPLAY RESULT
# ================================================================

print("=" * 70)
print("SIGNAL TEST #2 — CTR vs FUTURE DECAY")
print("=" * 70)

print("\nHypothesis:")
print("Lower CTR should be associated with higher future decay risk.")

print("\nDefinition:")
print("Future decay = future 3-month impressions decrease by >= 20%.")

print("\n" + "=" * 70)
print("CTR BUCKET RESULTS")
print("=" * 70)

display(
    ctr_results[
        [
            "ctr_bucket",
            "observations",
            "decay_cases",
            "decay_rate",
            "avg_future_impression_change_pct",
            "median_future_impression_change_pct"
        ]
    ]
)


# ================================================================
# 7. AUTOMATIC VERDICT
# ================================================================

valid = ctr_results.dropna(subset=["decay_rate"])

if len(valid) >= 2:

    lowest_ctr_decay = valid.iloc[0]["decay_rate"]
    highest_ctr_decay = valid.iloc[-1]["decay_rate"]

    # Expected:
    # Low CTR -> high decay
    # High CTR -> low decay

    if highest_ctr_decay < lowest_ctr_decay:
        verdict = "CONFIRMED"

    elif highest_ctr_decay > lowest_ctr_decay:
        verdict = "OPPOSITE"

    else:
        verdict = "MIXED"

    # Check whether decay generally decreases as CTR increases
    decay_values = valid["decay_rate"].values

    increasing = all(
        decay_values[i] <= decay_values[i + 1]
        for i in range(len(decay_values) - 1)
    )

    decreasing = all(
        decay_values[i] >= decay_values[i + 1]
        for i in range(len(decay_values) - 1)
    )

    if decreasing:
        verdict = "CONFIRMED"

    elif increasing:
        verdict = "OPPOSITE"

    else:
        verdict = "MIXED"

else:
    verdict = "FALSE"


# ================================================================
# 8. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("FINAL VERDICT")
print("=" * 70)

print(f"CTR observations tested : {len(df_ctr_test):,}")
print(f"VERDICT                  : {verdict}")

if verdict == "CONFIRMED":
    print(
        "Lower CTR is associated with higher future decay."
    )

elif verdict == "OPPOSITE":
    print(
        "Higher CTR is associated with higher future decay."
    )

elif verdict == "MIXED":
    print(
        "CTR shows no consistent monotonic relationship with future decay."
    )

else:
    print(
        "Insufficient usable data to evaluate the CTR signal."
    )

SIGNAL TEST #2 — CTR vs FUTURE DECAY

Hypothesis:
Lower CTR should be associated with higher future decay risk.

Definition:
Future decay = future 3-month impressions decrease by >= 20%.

CTR BUCKET RESULTS


,ctr_bucket,observations,decay_cases,decay_rate,avg_future_impression_change_pct,median_future_impression_change_pct
0,Very Low (<0.5%),751162,370297,49.296557,742.151395,-16.831956
1,Low (0.5–1%),65394,21512,32.895984,168.502048,27.895125
2,Medium (1–2%),34564,9738,28.173822,322.694558,58.948645
3,High (2–5%),16614,5015,30.185386,701.607153,95.961199
4,Very High (5%+),12371,6312,51.022553,1380.945240,-25.000000



FINAL VERDICT
CTR observations tested : 880,105
VERDICT                  : MIXED
CTR shows no consistent monotonic relationship with future decay.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.